In [1]:
import os
import sys
os.chdir('..')
current_dir = os.path.abspath('')
sys.path.append(current_dir)

In [2]:
import scripts.helpers
import scripts.base_helpers
from pytorch_lightning import seed_everything
import importlib
from tqdm import tqdm
import pickle

In [11]:
importlib.reload(scripts.helpers)
importlib.reload(scripts.base_helpers)
from scripts.helpers import *
from scripts.base_helpers import *

In [4]:
seed_everything(42)
SAVE_PATH = "outputs/sample_giuded_interp_test/"
set_lowvram_mode(False)
model = load_model_from_joblib("./checkpoints/sdxlbase1_cache.joblib")

Global seed set to 42


In [5]:
options = {
  "discretization": "LegacyDDPMDiscretization",
  "sigma_min"     : 0.03,   #EDMDiscretization, [-inf, inf | 0.03]
  "sigma_max"     : 14.61,  #EDMDiscretization, [-inf, inf | 14.61]
  "rho"           : 3.0,    #EDMDiscretization, [-inf, inf | 3.0]

  "guider"                  : "VanillaCFG",
  "additional_guider_kwargs": {},
  "vanilla_cfg"             : 2.0,  #VanillaCFG, [0.0, inf | 5.0]
  "linear_cfg"              : 1.5,  #LinearCFG, [1.0, inf | 1.5]
  "triangle_cfg"            : 2.5,  #TriangleCFG, [1.0, 10.0 | 2.5]
  "min_cfg"                 : 1.0,  #LinearCFG TriangleCFG, [1.0]
  "num_frames"              : 25,   #LinearCFG TriangleCFG, [25]

  "sampler"           : "HeunEDMSampler",
  "s_churn"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "s_tmin"            : 0.0,    #EulerEDM HeunEDM [0.0, inf |0.0]
  "s_tmax"            : 999.0,  #EulerEDM HeunEDM [0.0, inf | 999.0]
  "s_noise"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "eta"               : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "s_noise_ancestral" : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "order"             : 4,      #LinearMultistep [1, inf | 4]

  "crop_coords_top"         : 0,    #[0, inf | 0]
  "crop_coords_left"        : 0,    #[0, inf | 0]
  "aesthetic_score"         : 6.0,  #[-inf, inf | 6.0]
  "negative_aesthetic_score": 2.5,  #[-inf, inf | 2.5]
  "fps"                     : 6,    #[1, inf | 6]
  "mb_id"                   : 127,  #[0, 511 | 127]
  "image_path"              : None
}

In [12]:
prompts = read_prompts('./scripts/prompts.txt')
num_steps =  20
dims = [576, 1024] #height, width
sampler = init_sampling(options = options, steps = num_steps)

In [13]:
conds, ucs = get_conditionings(model, dims, prompts)

In [137]:
conds['crossattn'].shape

torch.Size([2, 77, 2048])

In [81]:
uc_single = {}
uc_single['crossattn'] = ucs['crossattn'][0:1]
uc_single['vector'] = ucs['vector'][0:1]

In [82]:
samples = get_samples(model, sampler, dims, conds, ucs)

##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 16 steps:  94%|█████████▍| 15/16 [00:04<00:00,  3.13it/s]


In [85]:
interpolated_sample = (samples[0] + samples[1]) / 2
interpolated_sample = interpolated_sample.unsqueeze(0)

In [92]:
interp_conds = {}
interp_conds['crossattn'] = conds['crossattn'].mean(dim=0, keepdim=True)
interp_conds['vector'] = conds['vector'].mean(dim=0, keepdim=True)

In [105]:
conds_all = {}
conds_all['crossattn'] = torch.stack([conds['crossattn'][0], interp_conds['crossattn'][0], conds['crossattn'][1]])
conds_all['vector'] = torch.stack([conds['vector'][0], interp_conds['vector'][0], conds['vector'][1]])

ucs_all = {}
ucs_all['crossattn'] = torch.stack([ucs['crossattn'][0]]*3)
ucs_all['vector'] = torch.stack([ucs['vector'][0]]*3)

In [106]:
interp_samples_all = get_samples(model, sampler, dims, conds_all, ucs_all)

##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 16 steps:  94%|█████████▍| 15/16 [00:07<00:00,  2.00it/s]


In [107]:
out = decode_in_chunks(model, interp_samples_all)

In [108]:
save_png(SAVE_PATH, out)

In [109]:
half_sample = interp_samples_all[0] * 0.5 + interp_samples_all[1] * 0.5
half_sample = half_sample.unsqueeze(0)

In [131]:
conds_test, ucs_test = get_conditionings(model, dims, [prompts[0]])
# test_sample = take_steps(model, sampler, conds_test, half_sample, steps=15)
test_sample = take_steps(model, sampler, conds_test, torch.randn_like(half_sample), steps=60)

In [132]:
out = decode_in_chunks(model, test_sample)
save_png(SAVE_PATH, out)

In [118]:
out = decode_in_chunks(model, half_sample)
save_png(SAVE_PATH, out)

In [8]:
interp_conds, interp_ucs = interpolate_conds(conds, ucs, interp=36)

In [9]:
interp_cond_ind = {}
interp_uc_ind = {}
cond_samples = []
for i in range(interp_conds['crossattn'].shape[0]):
    interp_cond_ind['crossattn'] = interp_conds['crossattn'][i].unsqueeze(0)
    interp_cond_ind['vector'] = interp_conds['vector'][i].unsqueeze(0)
    interp_uc_ind['crossattn'] = interp_ucs['crossattn'][i].unsqueeze(0)
    interp_uc_ind['vector'] = interp_ucs['vector'][i].unsqueeze(0)
    cond_sample_ind = get_samples(model, sampler, dims, interp_cond_ind, interp_uc_ind)
    cond_samples.append(cond_sample_ind)
cond_samples = torch.cat(cond_samples, dim=0)

##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 21 steps:  95%|█████████▌| 20/21 [00:04<00:00,  4.86it/s]


##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 21 steps:  95%|█████████▌| 20/21 [00:03<00:00,  5.28it/s]


##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


...

Sampling with HeunEDMSampler for 21 steps:  95%|█████████▌| 20/21 [00:03<00:00,  5.09it/s]


##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 21 steps:  95%|█████████▌| 20/21 [00:03<00:00,  5.23it/s]


##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 21 steps:  95%|█████████▌| 20/21 [00:03<00:00,  5.14it/s]


In [10]:
cond_samples_interpolated = interpolate_samples(cond_samples, interp=6)

In [12]:
with open("./scripts/cond_samples_interpolated.pkl", "wb") as f:
    pickle.dump(cond_samples_interpolated, f)

In [8]:
with open("./scripts/cond_samples_interpolated.pkl", "rb") as f:
    cond_samples_interpolated = pickle.load(f)

In [20]:
pooled_memory = torch.empty((1, 3, 576, 1024), device='cuda', dtype=torch.float32)

In [28]:
def decode_samples_memsafe(model, samples, pooled_memory):
    precision_scope = autocast
    with torch.no_grad():
        with precision_scope("cuda"):
            with model.ema_scope():
                samples_x = model.decode_first_stage(samples)
                samples = torch.clamp((samples_x + 1.0) / 2.0, min=0.0, max=1.0)
                # print(samples.shape)
                pooled_memory[:samples.shape[0], :samples.shape[1], :samples.shape[2], :samples.shape[3]] = samples
                grid = pooled_memory[:samples.shape[0]].unsqueeze(0)
                # grid = torch.stack([samples])
                # print(grid.shape)
                grid = rearrange(grid, "n b c h w -> c (n h) (b w)")
                return grid

In [12]:
outs = []
for ind_sample in tqdm(cond_samples_interpolated, desc="Decoding Samples"):
    clear_vram()
    out = decode_samples(model, ind_sample.unsqueeze(0))
    outs.append(out.detach().cpu())
    del out
out = torch.cat(outs, dim=0)

Decoding Samples: 100%|██████████| 2332/2332 [10:52<00:00,  3.57it/s]


In [17]:
save_mp4(SAVE_PATH, out, fps=24)

In [32]:
new_samples = interpolate_samples(samples, interp=1)

0.5


In [28]:
new_samples.shape

torch.Size([3, 4, 72, 128])

In [25]:
for i in range(len(new_samples[1:-1])):
    new_samples[i+1] = take_step(model, sampler, uc_single, new_samples[i+1:i+2])

TypeError: can't assign a NoneType to a torch.cuda.FloatTensor